In [ ]:
import ROOT, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [ ]:
dirDir= "/home/users/l/lebrun/"
dataDir = dirDir + "data/"
figPath = dirDir + "figures/scaling/"
resultPath = dirDir + "results/"
dataPath = dirDir + "data/"

file = dataPath + "MC_het_exp0.0_filtered_energy.csv"
rootfile = dataDir + "MC_exp0.0.root"
df = pd.read_csv(file)
rdf = ROOT.RDataFrame("MC", rootfile)

# print columns names
print("Columns in the DataFrame:")
for column in df.columns:
    print(column)

df["E_true"] = df["tt_ekin"] * 1e-3  # Convert from MeV to GeV
print(df["weight"])

In [ ]:
# Define log-spaced bins
bins = np.logspace(np.log10(1), np.log10(1e4), 50)

# Plot histogram with log-spaced bins
plt.figure(figsize=(6, 4))
plt.hist(df['E_true'], weights=df['weight'], bins=bins, color='blue', alpha=0.7, label='Energy Spectrum')

# Set log scales
plt.xscale('log')
plt.yscale('log')

# Labels and formatting
plt.xlabel('Energy (GeV)')
plt.ylabel('Counts')
plt.title('Energy Spectrum of Filtered MC Gamma Events')
plt.legend()
plt.grid(True, which="both", ls="--", linewidth=0.5)
plt.tight_layout()


In [ ]:
def create_log_bins(min_val, max_val, n_bins):
    """Create logarithmically spaced bin edges"""
    return np.logspace(np.log10(min_val), np.log10(max_val), n_bins + 1)

Emin = 1
Emax = 1e4
bounds = [100, 1000, Emax]
bounds_edges = [Emin] + bounds 
n_bins = 50 

# Create logarithmic bins
log_bins = create_log_bins(Emin, Emax, n_bins)
# Convert to ROOT format (ROOT needs array of doubles)
log_bins_root = ROOT.std.vector('double')(log_bins)

In [ ]:
# Weighted histogram  
h_weighted = rdf.Histo1D(ROOT.RDF.TH1DModel("h_weighted", "Power Law Weighted;Energy [GeV];Weighted Counts", 
                                                    n_bins, log_bins_root.data()), "E_true", "weight")


In [ ]:
%%cpp
double powerLaw(double *x, double a, double b){
    return b*pow(x[0], a);
}

In [ ]:
%%cpp
double fitFunction(double *x, double *par){
    return powerLaw(x, par[0], par[1]);
}

In [ ]:

# Plot weighted
c2 = ROOT.TCanvas()
ROOT.gPad.SetLogx()
ROOT.gPad.SetLogy() 
ROOT.gPad.SetGrid()

h_weight = h_weighted.GetPtr()
h_weight.SetLineColor(ROOT.kRed)
h_weight.SetLineWidth(2)
h_weight.GetXaxis().SetLabelOffset(0.005)
h_weight.GetXaxis().SetTitleOffset(1.3)
# Draw histogram
h_weight.Draw()

# perform the fit
fitFuncE = ROOT.TF1("fitFunction", ROOT.fitFunction, xmin=Emin, xmax=Emax, npar=2)
fitFuncE.SetParameter(0, -2.5) # a
fitFuncE.SetParameter(1, 4)    # b 
fitFuncE.SetLineColor(ROOT.kBlack)
fitFuncE.SetLineWidth(2)

# Fit with "R+" option to draw the fit function
res = h_weight.Fit(fitFuncE, "R")

# Add legend
alpha = fitFuncE.GetParameter(0)
beta = np.log10(fitFuncE.GetParameter(1))
leg = ROOT.TLegend(0.55, 0.7, 0.82, 0.8)
leg.AddEntry(fitFuncE, f"log_{{10}}(y) \propto {alpha:.2f} log_{{10}}(x)", "l")
leg.SetTextSize(0.03)
leg.Draw("same")

# Final update before saving
c2.Update()

In [ ]:
rdf = rdf.Define("cosTheta", "cos(theta_True * 180 / 3.141592653589793)")  # Convert radians to degrees
rdfs = [
    rdf.Filter(f"cosTheta >= {i * 0.05} && cosTheta < {(i + 1) * 0.05}")
    for i in range(20)
]
# Create histograms for each filter
hists = [rdfs[i].Histo1D(ROOT.RDF.TH1DModel(f"h_cosTheta_{i+1}", f"Cosine Theta < {0.05 * (i + 1)};Energy [GeV];Counts", n_bins, log_bins_root.data()), "E_true", "weight") for i in range(20)]

for i, r in enumerate(rdfs):
    print(f"Bin {i}: {r.Count().GetValue()} entries")
print(log_bins_root)
print(rdf.Max("E_true").GetValue(), rdf.Min("E_true").GetValue())

In [ ]:
# Draw histograms on same canvas for comparison
ROOT.gStyle.SetOptStat(0)
c3 = ROOT.TCanvas("c3", "Cosine Theta Histograms", 800, 600)
c3.SetLogy()
c3.SetLogx()
colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen+2, ROOT.kOrange, ROOT.kViolet, ROOT.kCyan+1,
          ROOT.kMagenta+2, ROOT.kGray+2, ROOT.kPink-3, ROOT.kTeal+3] * 2  # Repeat for 20
for i, hist in enumerate(hists):
    hist.SetLineColor(colors[i])
    hist.SetLineWidth(2)
    if i == 0:
        hist.Draw()  # Draw the first histogram
    else:
        hist.Draw("hist same")  # Draw subsequent histograms on the same canvas
# Add legend
leg2 = ROOT.TLegend(0.55, 0.65, 0.88, 0.88)
for i in range(20):
    leg2.AddEntry(hists[i].GetValue(), f"{0.05 * i:.2f} ≤ cosθ < {0.05 * (i + 1):.2f}", "l")
leg2.SetTextSize(0.03)
leg2.Draw("same")
  
# Update canvas
c3.Update()
# Show canvas
c3.Draw()